In [70]:
import torch
import gradio as gr 
import transformers  

In [31]:
# Use a pipeline as a high-level helper
#from transformers import pipeline
#pipe = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

In [32]:
model_path=("models/models--distilbert--distilbert-base-uncased-finetuned-sst-2-english/snapshots/714eb0fa89d2f80546fda750413ed43d93601a13")

In [33]:
analyzer = pipeline("text-classification", model=model_path)
print(analyzer(["this production is good","this product is quite expensive"]))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998596906661987}, {'label': 'NEGATIVE', 'score': 0.9986286163330078}]


In [59]:
def sentiment_analyzer(review):
    sentiment = analyzer(review)
    return sentiment[0]['label']

In [60]:
import matplotlib.pyplot as plt
def sentiment_bar_chart(df):
    sentiment_counts = df['Sentiment'].value_counts()

    # Create a bar chart
    fig, ax = plt.subplots()
    sentiment_counts.plot(kind='pie', ax=ax, autopct='%1.1f%%', color=['green', 'red'])
    ax.set_title('Review Sentiment Counts')
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Count')
    # ax.set_xticklabels(['Positive', 'Negative'], rotation=0)

    # Return the figure object
    return fig


In [61]:
import pandas as pd

def read_reviews_and_analyze_sentiment(file_object):
    # Load the Excel file into a DataFrame
    df = pd.read_excel(file_object)

    # Normalize column names: lowercase + strip spaces
    df.columns = df.columns.str.strip().str.lower()

    # Find a column containing 'review' (singular/plural, any case)
    review_col = None
    for col in df.columns:
        if 'review' in col:  # matches 'review', 'reviews', etc.
            review_col = col
            break

    if review_col is None:
        raise ValueError("Excel file must contain a column with 'review' in its name (any case, singular or plural).")

    # Apply the sentiment_analyzer function
    df['Sentiment'] = df[review_col].apply(sentiment_analyzer)

    # Create the chart
    chart_object = sentiment_bar_chart(df)

    return df, chart_object

In [62]:
result=read_reviews_and_analyze_sentiment("reviews.xlsx")
print(result)

(                                              review Sentiment
0  I absolutely love this new smartphone! The bat...  POSITIVE
1  This was the worst dining experience of my lif...  NEGATIVE
2  This fantasy novel is a masterpiece of world-b...  POSITIVE
3  I am extremely disappointed with this software...  NEGATIVE
4  Our stay at the Grand Hotel was absolutely per...  POSITIVE
5  I regret buying this jacket online. The materi...  NEGATIVE
6  This movie is a visual triumph with an emotion...  POSITIVE
7  Avoid this car repair shop at all costs. They ...  NEGATIVE
8  This air fryer has completely changed the way ...  POSITIVE
9  The laptop I purchased is a total disaster. It...  NEGATIVE, <Figure size 640x480 with 1 Axes>)


In [ ]:
custom_css = """ 
body {
    background: linear-gradient(135deg, #667eea, #764ba2);
}

/* Main container */
.gradio-container {
    border-radius: 20px;
    padding: 20px;
}

/* Title */
h1 {
    text-align: center;
    font-weight: 800;
    color: #ffffff;
}

/* Labels */
label {
    font-weight: bold;
    color: #ffffff;
}

/* INPUT / FILE BOX TEXT */
input, textarea {
    color: white !important;
}

/* FILE UPLOAD TEXT */
.file-preview, .file-name {
    color: white !important;
}

/* DATAFRAME TEXT */
table, th, td {
    color: white !important;
}

/* GENERAL COMPONENT TEXT */
.gr-box, .gr-input, .gr-output {
    color: white !important;
}
"""

# Gradio interface
demo = gr.Interface(
    fn=read_reviews_and_analyze_sentiment,
    inputs=[gr.File(file_types=[".xlsx"], label=" Upload your review file")],
    outputs=[
        gr.Dataframe(label=" Sentiment Results"),
        gr.Plot(label="Sentiment Visualization")
    ],
    title="✨ Review Sentiment Analyzer",
    description="Upload your Excel file and get instant sentiment insights.",
    theme=gr.themes.Soft(),
    css=custom_css
)

demo.launch()

c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
